# Feature Engineering v3 — Eventos Tácticos F1
**Unidad de análisis**: cada fila = un evento (overtake / undercut / overcut)

**Objetivo**: superar 500 variables para PCA significativo

| Bloque | Variables nuevas | Acumulado |
|---|---|---|
| Base v2 (heredado) | 205 | 205 |
| A: Tendencia temporal (slope, CV, range) | ~117 | ~322 |
| B: Ventana extendida (5 y 10 vueltas) | ~78 | ~400 |
| C: Ratios y eficiencias | ~40 | ~440 |
| D: Contexto de carrera | ~30 | ~470 |
| E: Degradación y estrategia | ~25 | ~495 |
| F: Encodings categóricos | ~20 | ~515 |

In [27]:
import polars as pl
import numpy as np
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import polars.selectors as cs

print(f'Polars: {pl.__version__}')
pl.Config.set_tbl_cols(20)

Polars: 1.35.2


polars.config.Config

In [28]:
# ==========================================
# CARGA DE DATOS  (ajustá BASE_DIR a tu ruta)
# ==========================================
carreras = ['australia_2026', 'china_2026', 'japan_2026', 'united_states_2026']
BASE_DIR  = Path('C:/Users/marce/F1-data-project/project/')

master_dfs = []
for c in carreras:
    p = BASE_DIR / 'data' / 'processed' / f'{c}_masterv2.parquet'
    if p.exists():
        # Leemos, convertimos ints a floats, y agregamos el nombre de la carrera
        df_temp = (
            pl.read_parquet(p)
            .with_columns(cs.integer().cast(pl.Float64))
            .with_columns(pl.lit(c).alias('race_name'))
        )
        master_dfs.append(df_temp)

# Concatenamos de forma diagonal por si alguna carrera tiene columnas extra (las 540)
df_master = pl.concat(master_dfs)

event_dfs = []
for c in carreras:
    p = BASE_DIR / 'data' / 'events' / f'{c}_events.parquet'
    if p.exists():
        # Aplicamos la misma estandarización para los eventos
        df_temp = (
            pl.read_parquet(p)
            .with_columns(cs.integer().cast(pl.Float64))
            .with_columns(pl.lit(c).alias('race_name'))
        )
        event_dfs.append(df_temp)

df_events = pl.concat(event_dfs)

print(f'Telemetría: {df_master.height} filas x {df_master.width} cols')
print(f'Eventos   : {df_events.height} eventos')

Telemetría: 3331 filas x 29 cols
Eventos   : 643 eventos


In [29]:
# ==========================================
# PARÁMETROS Y CONVERSIÓN DE TIPOS
# ==========================================
WINDOW_3  = 3
WINDOW_5  = 5
WINDOW_10 = 10

SENSOR_COLS = [
    'lap_duration',
    'duration_sector_1', 'duration_sector_2', 'duration_sector_3',
    'i1_speed', 'i2_speed', 'st_speed',
    'throttle_mean_lap', 'throttle_std_lap',
    'brake_max_lap', 'rpm_max_lap', 'n_gear_max_lap', 'drs_max_lap'
]

cols_existentes = [c for c in SENSOR_COLS if c in df_master.columns]
df_master = df_master.with_columns([
    pl.col(c).cast(pl.Float64, strict=False) for c in cols_existentes
])

# Columnas extra disponibles en master (posición, stint, etc.)
EXTRA_COLS = [c for c in ['position', 'stint_number', 'tyre_age', 'pit_duration',
                            'is_pit_lap', 'is_pit_out_lap']
              if c in df_master.columns]
df_master = df_master.with_columns([
    pl.col(c).cast(pl.Float64, strict=False) for c in EXTRA_COLS
])

print('Tipos aplicados OK')
print(f'Sensores disponibles: {cols_existentes}')

Tipos aplicados OK
Sensores disponibles: ['lap_duration', 'duration_sector_1', 'duration_sector_2', 'duration_sector_3', 'i1_speed', 'i2_speed', 'st_speed', 'throttle_mean_lap', 'throttle_std_lap', 'brake_max_lap', 'rpm_max_lap', 'n_gear_max_lap', 'drs_max_lap']


In [33]:
# ==========================================
# HELPERS
# ==========================================

def safe_item(df, expr):
    """Evalúa una expresión Polars y devuelve el valor escalar o None."""
    try:
        val = df.select(expr).item()
        return val
    except Exception:
        return None


def linreg_slope(values):
    """Pendiente de regresión lineal simple sobre una lista de valores."""
    if len(values) < 2:
        return None
    vals = [v for v in values if v is not None]
    if len(vals) < 2:
        return None
    x = np.arange(len(vals), dtype=float)
    y = np.array(vals, dtype=float)
    if np.std(x) == 0:
        return None
    return float(np.polyfit(x, y, 1)[0])


def safe_ratio(a, b):
    """División segura: devuelve None si b es 0 o None."""
    if a is None or b is None or b == 0:
        return None
    return a / b


def get_driver_window(df_race, driver, lap, window):
    """Devuelve las últimas `window` vueltas del piloto antes del evento."""
    # Envolvemos lap y window con int()
    target_laps = list(range(int(lap) - int(window), int(lap))) 
    
    return df_race.filter(
        (pl.col('driver_number') == driver) &
        (pl.col('lap_number').is_in(target_laps))
    )


def stats5(df, sensor):
    """Retorna dict con mean/max/min/std/median para un sensor dado."""
    if sensor not in df.columns or df.is_empty():
        return {s: None for s in ['mean','max','min','std','median']}
    col = pl.col(sensor)
    return {
        'mean':   safe_item(df, col.mean()),
        'max':    safe_item(df, col.max()),
        'min':    safe_item(df, col.min()),
        'std':    safe_item(df, col.std()),
        'median': safe_item(df, col.median()),
    }

In [34]:
# ==========================================
# FUNCIÓN PRINCIPAL DE FEATURE EXTRACTION
# ==========================================

def extract_all_features(event, df_master):
    race   = event.get('race_name', event.get('race_id'))
    lap    = event['lap_number']
    att    = event['initiator_driver']
    def_dr = event['target_driver']

    feat = {
        'event_id'  : f"{race}_L{lap}_{att}v{def_dr}",
        'race_name' : race,
        'lap_number': lap,
        'event_type': event.get('event_type'),
        'attacker'  : att,
        'defender'  : def_dr,
        'pos_change': event.get('initiator_pos_change'),
    }

    df_race = df_master.filter(pl.col('race_name') == race)

    # Ventanas de datos
    att3  = get_driver_window(df_race, att,    lap, WINDOW_3)
    def3  = get_driver_window(df_race, def_dr, lap, WINDOW_3)
    att5  = get_driver_window(df_race, att,    lap, WINDOW_5)
    def5  = get_driver_window(df_race, def_dr, lap, WINDOW_5)
    att10 = get_driver_window(df_race, att,    lap, WINDOW_10)
    def10 = get_driver_window(df_race, def_dr, lap, WINDOW_10)

    # Vuelta exactamente anterior
    att_prev = df_race.filter((pl.col('driver_number') == att)    & (pl.col('lap_number') == lap - 1))
    def_prev = df_race.filter((pl.col('driver_number') == def_dr) & (pl.col('lap_number') == lap - 1))

    # ── BLOQUE 0: Info táctica base ──────────────────────────────────────────
    # Compuesto
    for prefix, src in [('att', att_prev), ('def', def_prev)]:
        feat[f'{prefix}_compound'] = safe_item(src, pl.col('compound')) if 'compound' in src.columns else None
        feat[f'{prefix}_tyre_age'] = safe_item(src, pl.col('tyre_age')) if 'tyre_age' in src.columns else None
        feat[f'{prefix}_stint_number'] = safe_item(src, pl.col('stint_number')) if 'stint_number' in src.columns else None
        feat[f'{prefix}_position']    = safe_item(src, pl.col('position'))    if 'position'     in src.columns else None

    if feat['att_tyre_age'] is not None and feat['def_tyre_age'] is not None:
        feat['delta_tyre_age'] = feat['att_tyre_age'] - feat['def_tyre_age']
    else:
        feat['delta_tyre_age'] = None

    feat['position_gap'] = (
        feat['def_position'] - feat['att_position']
        if feat['att_position'] is not None and feat['def_position'] is not None
        else None
    )

    # ── BLOQUE A: Stats sobre ventana de 3 vueltas (heredado + limpio) ───────
    # mean / max / min / std / median  +  delta  = 15 cols por sensor
    for sensor in cols_existentes:
        a = stats5(att3, sensor)
        d = stats5(def3, sensor)
        for stat, val in a.items():
            feat[f'att_{sensor}_{stat}'] = val
        for stat, val in d.items():
            feat[f'def_{sensor}_{stat}'] = val
        for stat in ['mean','max','min','std','median']:
            feat[f'delta_{sensor}_{stat}'] = (
                a[stat] - d[stat] if a[stat] is not None and d[stat] is not None else None
            )

    # ── BLOQUE A2: Tendencia (slope) y variabilidad (CV, range) en 3 vueltas ─
    # +3 cols por sensor x 13 sensores x 3 roles (att/def/delta) = ~117 cols
    for sensor in cols_existentes:
        for prefix, src in [('att', att3), ('def', def3)]:
            vals = src[sensor].to_list() if sensor in src.columns and not src.is_empty() else []
            vals = [v for v in vals if v is not None]

            slope = linreg_slope(vals)
            feat[f'{prefix}_{sensor}_slope']  = slope

            mean_v = np.mean(vals) if vals else None
            std_v  = np.std(vals)  if vals else None
            feat[f'{prefix}_{sensor}_cv']     = safe_ratio(std_v, mean_v)   # Coef. variación
            feat[f'{prefix}_{sensor}_range']  = (max(vals) - min(vals)) if len(vals) >= 2 else None

        # Delta slope (¿quién está mejorando más rápido?)
        feat[f'delta_{sensor}_slope'] = (
            feat[f'att_{sensor}_slope'] - feat[f'def_{sensor}_slope']
            if feat[f'att_{sensor}_slope'] is not None and feat[f'def_{sensor}_slope'] is not None
            else None
        )

    # ── BLOQUE B: Ventana de 5 vueltas ───────────────────────────────────────
    # mean + std + slope = 3 stats x 13 sensores x 3 roles = ~78 cols
    for sensor in cols_existentes:
        for prefix, src in [('att', att5), ('def', def5)]:
            s = stats5(src, sensor)
            feat[f'{prefix}_{sensor}_w5_mean'] = s['mean']
            feat[f'{prefix}_{sensor}_w5_std']  = s['std']

            vals = src[sensor].to_list() if sensor in src.columns and not src.is_empty() else []
            feat[f'{prefix}_{sensor}_w5_slope'] = linreg_slope([v for v in vals if v is not None])

        feat[f'delta_{sensor}_w5_mean'] = (
            feat[f'att_{sensor}_w5_mean'] - feat[f'def_{sensor}_w5_mean']
            if feat[f'att_{sensor}_w5_mean'] is not None and feat[f'def_{sensor}_w5_mean'] is not None
            else None
        )

    # ── BLOQUE B2: Ventana de 10 vueltas (solo mean y slope) ─────────────────
    for sensor in cols_existentes:
        for prefix, src in [('att', att10), ('def', def10)]:
            s = stats5(src, sensor)
            feat[f'{prefix}_{sensor}_w10_mean'] = s['mean']
            vals = src[sensor].to_list() if sensor in src.columns and not src.is_empty() else []
            feat[f'{prefix}_{sensor}_w10_slope'] = linreg_slope([v for v in vals if v is not None])

        feat[f'delta_{sensor}_w10_mean'] = (
            feat[f'att_{sensor}_w10_mean'] - feat[f'def_{sensor}_w10_mean']
            if feat[f'att_{sensor}_w10_mean'] is not None and feat[f'def_{sensor}_w10_mean'] is not None
            else None
        )

    # ── BLOQUE C: Ratios y eficiencias ───────────────────────────────────────
    # ~40 variables

    # C1: Proporción sectorial del tiempo total (cuánto % del lap es cada sector)
    for prefix in ['att', 'def']:
        lap_m = feat.get(f'{prefix}_lap_duration_mean')
        for s_num in [1, 2, 3]:
            sec_m = feat.get(f'{prefix}_duration_sector_{s_num}_mean')
            feat[f'{prefix}_sector{s_num}_pct'] = safe_ratio(sec_m, lap_m)

    for s_num in [1, 2, 3]:
        feat[f'delta_sector{s_num}_pct'] = (
            feat[f'att_sector{s_num}_pct'] - feat[f'def_sector{s_num}_pct']
            if feat[f'att_sector{s_num}_pct'] is not None and feat[f'def_sector{s_num}_pct'] is not None
            else None
        )

    # C2: Eficiencia velocidad / RPM
    for prefix in ['att', 'def']:
        spd = feat.get(f'{prefix}_st_speed_mean')
        rpm = feat.get(f'{prefix}_rpm_max_lap_mean')
        thr = feat.get(f'{prefix}_throttle_mean_lap_mean')
        brk = feat.get(f'{prefix}_brake_max_lap_mean')
        lap_m = feat.get(f'{prefix}_lap_duration_mean')

        feat[f'{prefix}_speed_per_rpm']         = safe_ratio(spd, rpm)
        feat[f'{prefix}_throttle_brake_ratio']  = safe_ratio(thr, brk)
        feat[f'{prefix}_speed_consistency']     = safe_ratio(
            feat.get(f'{prefix}_st_speed_std'), spd
        )  # Menor = más consistente
        feat[f'{prefix}_sector_balance']        = safe_ratio(
            feat.get(f'{prefix}_duration_sector_1_mean'),
            feat.get(f'{prefix}_duration_sector_3_mean')
        )  # S1/S3: si > 1 → débil en S1 vs S3
        feat[f'{prefix}_pace_per_tyre_age']     = safe_ratio(lap_m, feat[f'{prefix}_tyre_age'])

    for col in ['speed_per_rpm', 'throttle_brake_ratio', 'speed_consistency',
                'sector_balance', 'pace_per_tyre_age']:
        feat[f'delta_{col}'] = (
            feat[f'att_{col}'] - feat[f'def_{col}']
            if feat.get(f'att_{col}') is not None and feat.get(f'def_{col}') is not None
            else None
        )

    # C3: Comparativa w3 vs w10 (¿está mejorando o empeorando respecto al promedio largo?)
    for sensor in ['lap_duration', 'st_speed', 'throttle_mean_lap']:
        if sensor not in cols_existentes:
            continue
        for prefix in ['att', 'def']:
            m3  = feat.get(f'{prefix}_{sensor}_mean')
            m10 = feat.get(f'{prefix}_{sensor}_w10_mean')
            feat[f'{prefix}_{sensor}_momentum'] = safe_ratio(m3, m10)  # < 1 = mejorando

        feat[f'delta_{sensor}_momentum'] = (
            feat[f'att_{sensor}_momentum'] - feat[f'def_{sensor}_momentum']
            if feat.get(f'att_{sensor}_momentum') is not None and feat.get(f'def_{sensor}_momentum') is not None
            else None
        )

    # ── BLOQUE D: Contexto de carrera ─────────────────────────────────────────
    # ~30 variables

    # D1: Fase de carrera
    total_laps_series = df_race.select(pl.col('lap_number').max())
    total_laps = safe_item(total_laps_series, pl.col('lap_number'))
    feat['race_pct_complete'] = safe_ratio(lap, total_laps)
    feat['is_early_race']     = int(feat['race_pct_complete'] < 0.33) if feat['race_pct_complete'] is not None else None
    feat['is_mid_race']       = int(0.33 <= feat['race_pct_complete'] < 0.66) if feat['race_pct_complete'] is not None else None
    feat['is_late_race']      = int(feat['race_pct_complete'] >= 0.66) if feat['race_pct_complete'] is not None else None
    feat['laps_remaining']    = total_laps - lap if total_laps is not None else None

    # D2: Posición relativa en carrera
    feat['att_pos_pct']  = safe_ratio(feat.get('att_position'), 20)  # aprox 20 coches
    feat['def_pos_pct']  = safe_ratio(feat.get('def_position'), 20)
    feat['is_top10_battle'] = (
        int(feat['att_position'] <= 10 and feat['def_position'] <= 10)
        if feat.get('att_position') is not None and feat.get('def_position') is not None
        else None
    )

    # D3: Historial de eventos previos del atacante en la misma carrera
    att_prev_events = df_events.filter(
        (pl.col('race_name') == race) &
        (pl.col('initiator_driver') == att) &
        (pl.col('lap_number') < lap)
    ) if 'race_name' in df_events.columns else pl.DataFrame()

    feat['att_prev_events_count'] = att_prev_events.height
    feat['att_laps_since_last_event'] = (
        lap - att_prev_events.select(pl.col('lap_number').max()).item()
        if att_prev_events.height > 0 else None
    )

    # D4: ¿Hay SC/VSC reciente? (si hay lap_duration anómalo en los últimos 3 laps del campo)
    field_lap3 = df_race.filter(pl.col('lap_number').is_in([lap-1, lap-2, lap-3]))
    if not field_lap3.is_empty() and 'lap_duration' in field_lap3.columns:
        median_field = safe_item(field_lap3, pl.col('lap_duration').median())
        mean_field   = safe_item(field_lap3, pl.col('lap_duration').mean())
        feat['field_lap_duration_median'] = median_field
        feat['field_lap_duration_cv']     = safe_ratio(
            safe_item(field_lap3, pl.col('lap_duration').std()), mean_field
        )
        feat['possible_sc_laps']          = int(feat['field_lap_duration_cv'] > 0.05) if feat['field_lap_duration_cv'] is not None else None
    else:
        feat['field_lap_duration_median'] = None
        feat['field_lap_duration_cv']     = None
        feat['possible_sc_laps']          = None

    # D5: Pace relativo vs el campo (¿atacante es rápido vs todos?)
    if feat['field_lap_duration_median'] is not None:
        for prefix in ['att', 'def']:
            pace_m = feat.get(f'{prefix}_lap_duration_mean')
            feat[f'{prefix}_pace_vs_field'] = safe_ratio(pace_m, feat['field_lap_duration_median'])
        feat['delta_pace_vs_field'] = (
            feat.get('att_pace_vs_field') - feat.get('def_pace_vs_field')
            if feat.get('att_pace_vs_field') is not None and feat.get('def_pace_vs_field') is not None
            else None
        )

    # ── BLOQUE E: Degradación y estrategia de neumáticos ─────────────────────
    # ~25 variables

    # E1: Tasa de degradación (slope del lap_duration en el stint actual)
    for prefix, att_flag in [('att', att), ('def', def_dr)]:
        stint = feat.get(f'{prefix}_stint_number')
        if stint is not None and 'stint_number' in df_race.columns:
            stint_data = df_race.filter(
                (pl.col('driver_number') == att_flag) &
                (pl.col('stint_number') == int(stint)) &
                (pl.col('lap_number') < lap)
            )
            if not stint_data.is_empty() and 'lap_duration' in stint_data.columns:
                vals = stint_data['lap_duration'].to_list()
                vals = [v for v in vals if v is not None]
                feat[f'{prefix}_deg_rate']        = linreg_slope(vals)
                feat[f'{prefix}_stint_laps_done'] = len(vals)
                feat[f'{prefix}_best_lap_stint']  = min(vals) if vals else None
                feat[f'{prefix}_worst_lap_stint'] = max(vals) if vals else None
                feat[f'{prefix}_pace_drop']       = (
                    max(vals) - min(vals) if len(vals) >= 2 else None
                )
            else:
                for col in ['deg_rate','stint_laps_done','best_lap_stint','worst_lap_stint','pace_drop']:
                    feat[f'{prefix}_{col}'] = None
        else:
            for col in ['deg_rate','stint_laps_done','best_lap_stint','worst_lap_stint','pace_drop']:
                feat[f'{prefix}_{col}'] = None

    feat['delta_deg_rate']        = (
        feat.get('att_deg_rate') - feat.get('def_deg_rate')
        if feat.get('att_deg_rate') is not None and feat.get('def_deg_rate') is not None else None
    )
    feat['delta_stint_laps_done'] = (
        feat.get('att_stint_laps_done') - feat.get('def_stint_laps_done')
        if feat.get('att_stint_laps_done') is not None and feat.get('def_stint_laps_done') is not None else None
    )
    feat['delta_pace_drop'] = (
        feat.get('att_pace_drop') - feat.get('def_pace_drop')
        if feat.get('att_pace_drop') is not None and feat.get('def_pace_drop') is not None else None
    )

    # E2: ¿El atacante está en pit_out? (ventaja temporal de neumático fresco)
    feat['att_is_pit_out'] = (
        int(safe_item(att_prev, pl.col('is_pit_out_lap')) == 1.0)
        if 'is_pit_out_lap' in att_prev.columns and not att_prev.is_empty() else 0
    )
    feat['def_is_pit_out'] = (
        int(safe_item(def_prev, pl.col('is_pit_out_lap')) == 1.0)
        if 'is_pit_out_lap' in def_prev.columns and not def_prev.is_empty() else 0
    )

    # ── BLOQUE F: Encodings de variables categóricas ──────────────────────────
    # ~20 variables

    # F1: One-hot de compuesto (SOFT/MEDIUM/HARD)
    compounds = ['SOFT', 'MEDIUM', 'HARD', 'INTERMEDIATE', 'WET']
    for prefix in ['att', 'def']:
        comp = feat.get(f'{prefix}_compound')
        for c in compounds:
            feat[f'{prefix}_is_{c.lower()}'] = int(comp == c) if comp is not None else 0

    # F2: Encoding del tipo de evento
    event_types = ['On_Track_Overtake', 'Undercut', 'Overcut']
    etype = feat.get('event_type')
    for et in event_types:
        feat[f'is_{et.lower()}'] = int(etype == et) if etype is not None else 0

    # F3: Ventaja de compuesto (nominal → ordinal)
    compound_order = {'SOFT': 1, 'MEDIUM': 2, 'HARD': 3, 'INTERMEDIATE': 1.5, 'WET': 0.5}
    att_c_val = compound_order.get(feat.get('att_compound'), None)
    def_c_val = compound_order.get(feat.get('def_compound'), None)
    feat['att_compound_ord'] = att_c_val
    feat['def_compound_ord'] = def_c_val
    feat['delta_compound_ord'] = (
        att_c_val - def_c_val if att_c_val is not None and def_c_val is not None else None
    )

    # F4: Carrera encoding
    race_map = {'australia_2026': 0, 'china_2026': 1, 'japan_2026': 2}
    feat['race_encoded'] = race_map.get(race, -1)

    return feat

In [35]:
# ==========================================
# EJECUCIÓN
# ==========================================
print('⚙️  Extrayendo features... (esto puede tardar ~2-3 min por el slope de 10 vueltas)')

features_list = []
for i, event_row in enumerate(df_events.to_dicts()):
    
    res = extract_all_features(event_row, df_master)
    features_list.append(res)
    if (i + 1) % 50 == 0:
        print(f'  {i+1}/{df_events.height} eventos procesados...')

import pandas as pd
df_feat = pd.DataFrame(features_list)

print(f'\n✅ COMPLETADO')
print(f'   Eventos   : {len(df_feat)}')
print(f'   Variables  : {len(df_feat.columns)}')
df_feat.head(3)

⚙️  Extrayendo features... (esto puede tardar ~2-3 min por el slope de 10 vueltas)
  50/643 eventos procesados...
  100/643 eventos procesados...
  150/643 eventos procesados...
  200/643 eventos procesados...
  250/643 eventos procesados...
  300/643 eventos procesados...
  350/643 eventos procesados...
  400/643 eventos procesados...
  450/643 eventos procesados...
  500/643 eventos procesados...
  550/643 eventos procesados...
  600/643 eventos procesados...

✅ COMPLETADO
   Eventos   : 643
   Variables  : 540


,event_id,race_name,lap_number,event_type,attacker,defender,pos_change,att_compound,att_tyre_age,att_stint_number,...,def_is_hard,def_is_intermediate,def_is_wet,is_on_track_overtake,is_undercut,is_overcut,att_compound_ord,def_compound_ord,delta_compound_ord,race_encoded
0,australia_2026_L4.0_1.0v10.0,australia_2026,4.0,On_Track_Overtake,1.0,10.0,P10 -> P7,MEDIUM,2.0,1.0,...,0,0,0,1,0,0,2.0,NaN,NaN,0
1,australia_2026_L4.0_1.0v87.0,australia_2026,4.0,On_Track_Overtake,1.0,87.0,P10 -> P7,MEDIUM,2.0,1.0,...,0,0,0,1,0,0,2.0,2.0,0.0,0
2,australia_2026_L4.0_1.0v5.0,australia_2026,4.0,On_Track_Overtake,1.0,5.0,P10 -> P7,MEDIUM,2.0,1.0,...,0,0,0,1,0,0,2.0,2.0,0.0,0


In [36]:
# ==========================================
# DIAGNÓSTICO DE NULOS
# ==========================================
null_pct = df_feat.isnull().mean().sort_values(ascending=False)
print('Columnas con > 50% nulos (candidatas a eliminar):')
print(null_pct[null_pct > 0.5].to_string())
print(f'\nColumnas OK (< 50% nulos): {(null_pct <= 0.5).sum()}')

Columnas con > 50% nulos (candidatas a eliminar):
delta_drs_max_lap_w10_mean    1.0
att_drs_max_lap_w10_mean      1.0
att_drs_max_lap_w10_slope     1.0
def_drs_max_lap_w10_mean      1.0
def_drs_max_lap_w10_slope     1.0
att_drs_max_lap_cv            1.0
att_drs_max_lap_w5_slope      1.0
att_drs_max_lap_w5_std        1.0
att_drs_max_lap_w5_mean       1.0
def_drs_max_lap_w5_mean       1.0
def_drs_max_lap_w5_slope      1.0
delta_drs_max_lap_w5_mean     1.0
def_drs_max_lap_w5_std        1.0
att_drs_max_lap_slope         1.0
att_drs_max_lap_range         1.0
def_drs_max_lap_cv            1.0
delta_drs_max_lap_slope       1.0
def_drs_max_lap_slope         1.0
def_drs_max_lap_range         1.0
att_drs_max_lap_min           1.0
delta_drs_max_lap_median      1.0
att_drs_max_lap_median        1.0
def_drs_max_lap_mean          1.0
delta_drs_max_lap_mean        1.0
def_drs_max_lap_min           1.0
def_drs_max_lap_std           1.0
def_drs_max_lap_median        1.0
delta_drs_max_lap_max         1.

In [37]:
# ==========================================
# EXPORTACIÓN
# ==========================================
out_path = BASE_DIR / 'data' / 'features' / 'tactical_events_v3.parquet'
out_path.parent.mkdir(parents=True, exist_ok=True)

df_feat_pl = pl.from_pandas(df_feat)
df_feat_pl.write_parquet(str(out_path))

print(f'💾 Guardado en: {out_path}')
print(f'   Shape final: {df_feat.shape}')
print(f'\nResumen de bloques:')

bloques = {
    'Base (id, tipo, contexto)': [c for c in df_feat.columns if c in ['event_id','race_name','lap_number','event_type','attacker','defender','pos_change','delta_tyre_age','position_gap']],
    'Bloque A (stats 3 vueltas)': [c for c in df_feat.columns if any(s in c for s in ['_mean','_max','_min','_std','_median']) and '_w5_' not in c and '_w10_' not in c],
    'Bloque A2 (slope/CV/range)': [c for c in df_feat.columns if any(s in c for s in ['_slope','_cv','_range'])],
    'Bloque B (ventana 5v)':  [c for c in df_feat.columns if '_w5_' in c],
    'Bloque B2 (ventana 10v)': [c for c in df_feat.columns if '_w10_' in c],
    'Bloque C (ratios)': [c for c in df_feat.columns if any(s in c for s in ['_pct','_ratio','_per_','_balance','momentum','_consistency'])],
    'Bloque D (contexto carrera)': [c for c in df_feat.columns if any(s in c for s in ['race_pct','is_early','is_mid','is_late','laps_rem','top10','prev_events','field_'])],
    'Bloque E (degradación)': [c for c in df_feat.columns if any(s in c for s in ['deg_rate','stint_laps','best_lap','worst_lap','pace_drop','pit_out'])],
    'Bloque F (encodings)': [c for c in df_feat.columns if any(s in c for s in ['is_soft','is_medium','is_hard','is_on_track','is_undercut','is_overcut','compound_ord','race_encoded'])],
}

total = 0
for bloque, cols in bloques.items():
    print(f'  {bloque}: {len(cols)} cols')
    total += len(cols)
print(f'  TOTAL clasificadas: {total} / {len(df_feat.columns)}')

💾 Guardado en: C:\Users\marce\F1-data-project\project\data\features\tactical_events_v3.parquet
   Shape final: (643, 540)

Resumen de bloques:
  Base (id, tipo, contexto): 9 cols
  Bloque A (stats 3 vueltas): 241 cols
  Bloque A2 (slope/CV/range): 144 cols
  Bloque B (ventana 5v): 91 cols
  Bloque B2 (ventana 10v): 65 cols
  Bloque C (ratios): 36 cols
  Bloque D (contexto carrera): 9 cols
  Bloque E (degradación): 15 cols
  Bloque F (encodings): 13 cols
  TOTAL clasificadas: 623 / 540


In [38]:
# ==========================================
# PREPARACIÓN PARA PCA (preview)
# ==========================================
# Columnas no numéricas a excluir
id_cols = ['event_id', 'race_name', 'event_type', 'attacker', 'defender',
           'pos_change', 'att_compound', 'def_compound']

X = df_feat.drop(columns=[c for c in id_cols if c in df_feat.columns])
X = X.select_dtypes(include='number')

# Imputar medianas y eliminar columnas con > 50% nulos
null_pct = X.isnull().mean()
X = X.loc[:, null_pct <= 0.5]
X = X.fillna(X.median())

print(f'Features para PCA: {X.shape[1]} columnas x {X.shape[0]} filas')

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=min(20, X_scaled.shape[1]))
pca.fit(X_scaled)

var_exp = pca.explained_variance_ratio_.cumsum()
n_90 = next((i+1 for i, v in enumerate(var_exp) if v >= 0.90), len(var_exp))
print(f'Varianza explicada acumulada:')
for i, v in enumerate(var_exp[:10]):
    print(f'  PC{i+1}: {v:.1%}')
print(f'  → {n_90} componentes explican el 90% de la varianza')

Features para PCA: 498 columnas x 643 filas
Varianza explicada acumulada:
  PC1: 26.1%
  PC2: 35.9%
  PC3: 41.6%
  PC4: 45.5%
  PC5: 49.2%
  PC6: 52.0%
  PC7: 54.6%
  PC8: 56.9%
  PC9: 59.1%
  PC10: 61.1%
  → 20 componentes explican el 90% de la varianza


In [39]:
df_2 = pl.read_parquet('C:\\Users\\marce\\F1-data-project\\project\\data\\features\\tactical_events_v3.parquet')

In [40]:
df_2.shape

(643, 540)